# ArtGAN-GP &mdash; Google Colab driver

Open this notebook in Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/assylbek-creation/ArtGAN-GP/blob/main/notebooks/colab_train.ipynb)

This notebook is a thin driver for the modular code that lives in [the GitHub repo](https://github.com/assylbek-creation/ArtGAN-GP). It clones the repo, installs deps, downloads the dataset, trains the model, and visualizes the results &mdash; without copying any logic out of the repo.

**Before you run anything:**

1. `Runtime` &rarr; `Change runtime type` &rarr; `T4 GPU` &rarr; `Save`.
2. Pick a `MODE` in the next cell:
   - `'smoke'` &mdash; 500 images, 5 epochs (~10 minutes). Use this first to confirm everything works.
   - `'full'` &mdash; entire abstract WikiArt subset, 100 epochs (~30&ndash;50 minutes on T4). Use this for the final run.
3. `Runtime` &rarr; `Run all`.

When prompted, paste your W&B API key (https://wandb.ai/authorize) and optionally a HuggingFace token (https://huggingface.co/settings/tokens).

## 1. Mode + GPU check

In [ ]:
MODE = 'smoke'  # 'smoke' or 'full'
RUN_SWEEP = False  # set True to launch a 5-trial Bayesian sweep after training (~80 min)

import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv']).decode())

## 2. Clone repository

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/assylbek-creation/ArtGAN-GP.git'
REPO_DIR = Path('/content/ArtGAN-GP')

if REPO_DIR.exists():
    print(f'{REPO_DIR} already exists; pulling latest.')
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('CWD:', os.getcwd())

## 3. Install dependencies

Colab already ships with PyTorch + CUDA, so we just add the project-specific packages.

In [ ]:
!pip install -q hydra-core omegaconf wandb datasets torchmetrics[image] tqdm Pillow

## 4. Authentication

Both prompts are optional but recommended:
- **W&B API key** &mdash; required if `wandb.mode=online`. Skip by leaving blank to fall back to print-only logging.
- **HuggingFace token** &mdash; helps avoid rate limits when streaming WikiArt; not strictly required.

In [ ]:
import getpass

wb_key = getpass.getpass('W&B API key (blank to disable W&B): ').strip()
if wb_key:
    os.environ['WANDB_API_KEY'] = wb_key
    WANDB_MODE = 'online'
    print('W&B online.')
else:
    WANDB_MODE = 'disabled'
    print('W&B disabled; logs go to stdout only.')

hf_token = getpass.getpass('HuggingFace token (blank to skip): ').strip()
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Logged into HuggingFace.')

## 5. Download data

`scripts/download_data.py` streams `huggan/wikiart`, filters to the genres in `src/config/data/wikiart_abstract.yaml`, center-crops to 64x64, and saves PNGs.

In [ ]:
MAX_IMAGES = 500 if MODE == 'smoke' else 'null'
!python -m scripts.download_data data.max_images={MAX_IMAGES}

## 6. Train

We call `src.training.train.run` directly (the same entry point the W&B sweep uses), so all of the modular code in the repo &mdash; data loaders, models, training loop, gradient penalty, checkpointing &mdash; is exercised exactly as it would be on the command line.

In [ ]:
from hydra import compose, initialize_config_dir
from src.training.train import run

if MODE == 'smoke':
    overrides = [
        'training.epochs=5',
        'training.checkpoint_every_n_epochs=5',
        'training.fid_every_n_epochs=5',
        'training.fid_num_samples=500',
        'data.batch_size=64',
    ]
else:
    overrides = [
        'training.epochs=100',
        'training.checkpoint_every_n_epochs=10',
        'training.fid_every_n_epochs=10',
        'training.fid_num_samples=2000',
        'data.batch_size=64',
    ]
overrides.append(f'wandb.mode={WANDB_MODE}')

with initialize_config_dir(config_dir=str(REPO_DIR / 'src' / 'config'), version_base=None):
    cfg = compose(config_name='baseline', overrides=overrides)

print(f'Training in {MODE} mode for {cfg.training.epochs} epochs.')
run(cfg)

## 7. Visualize the latest checkpoint

Pick the most recent checkpoint, render an 8x8 random grid and a 10-frame slerp interpolation between two seeds, and display them inline.

In [ ]:
import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from src.models import build_models
from src.utils.checkpoint import load_checkpoint
from src.utils.interpolation import slerp_path
from src.utils.visualize import make_sample_grid

ckpts = sorted(Path('checkpoints').glob('epoch_*.pt'))
if not ckpts:
    raise FileNotFoundError('No checkpoints under ./checkpoints; did training finish?')
ckpt_path = ckpts[-1]
print(f'Loading {ckpt_path}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state = torch.load(str(ckpt_path), map_location=device)
ckpt_cfg = OmegaConf.create(state['config'])
generator, critic = build_models(ckpt_cfg)
generator.to(device).eval()
critic.to(device).eval()
load_checkpoint(ckpt_path, generator=generator, critic=critic, map_location=device)

def show(grid, title, figsize=(8, 8)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
    ax.axis('off')
    ax.set_title(title)
    plt.show()

with torch.no_grad():
    torch.manual_seed(0)
    z = torch.randn(64, ckpt_cfg.model.latent_dim, device=device)
    samples = generator(z)
show(make_sample_grid(samples, nrow=8), f'Random samples ({ckpt_path.name})')

def seeded_z(seed):
    g = torch.Generator(device=device).manual_seed(seed)
    return torch.randn(ckpt_cfg.model.latent_dim, device=device, generator=g)

with torch.no_grad():
    path = slerp_path(seeded_z(0), seeded_z(7), n_steps=10)
    interp = generator(path)
show(make_sample_grid(interp, nrow=10), 'Slerp interpolation seed 0 -> 7', figsize=(14, 2))

## 8. (Optional) Bayesian sweep

Set `RUN_SWEEP = True` in the first cell to launch a small W&B Bayesian sweep over the WGAN-GP hyperparameters defined in `src/config/sweep.yaml` (LR for generator and critic, batch size, lambda_gp, n_critic). Five trials at 30 epochs each takes roughly 80 minutes on a T4.

In [ ]:
if RUN_SWEEP and WANDB_MODE == 'online':
    !python -m scripts.run_sweep --count 5
else:
    print('Skipping sweep (RUN_SWEEP is False or W&B disabled).')

## 9. (Optional) Persist checkpoints to Google Drive

Colab wipes `/content/` between sessions. Mount Drive and copy the `checkpoints/` and `outputs/samples/` directories there if you want to keep them around.

In [ ]:
PERSIST = False  # set True to copy artifacts to Drive
if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    target = Path('/content/drive/MyDrive/ArtGAN-GP')
    target.mkdir(parents=True, exist_ok=True)
    !cp -r checkpoints {target}/
    !cp -r outputs {target}/ 2>/dev/null || true
    print(f'Copied checkpoints + outputs to {target}')
else:
    print('Persistence skipped (set PERSIST=True to enable).')